# 04. 공통 평가 질문 실행

OpenAI와 Local Retriever 완성 후 같은 질문 세트를 실행해 비교합니다. 이 노트북은 실행 결과를 포함하지 않습니다.


In [ ]:
import json
import sys
import time
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "api_main.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
QUESTION_PATH = PROJECT_ROOT / "data" / "evaluation" / "questions.jsonl"

with QUESTION_PATH.open(encoding="utf-8") as file:
    questions = [json.loads(line) for line in file]

display(pd.DataFrame(questions)[["question_id", "type", "question", "expected_doc_ids"]])


In [ ]:
# OpenAI와 Local 구현 완료 후 아래 셀을 실행합니다.
sys.path.insert(0, str(PROJECT_ROOT))
from api_main import run

PROFILE = "openai"  # 실행 시 openai 또는 local 중 하나를 선택

records = []
history_by_conversation = {}
for item in questions:
    conversation_id = item.get("conversation_id")
    history = history_by_conversation.get(conversation_id)
    started = time.perf_counter()
    response = run(item["question"], profile=PROFILE, filters=item.get("filters"), history=history)
    elapsed = time.perf_counter() - started
    if conversation_id:
        history_by_conversation.setdefault(conversation_id, []).append(
            {"question": item["question"], "answer": response["answer"]}
        )
    records.append({
        "question_id": item["question_id"],
        "profile": PROFILE,
        "question": item["question"],
        "retrieved_doc_ids": [source.get("doc_id") for source in response["sources"]],
        "final_answer": response["answer"],
        "response_seconds": round(elapsed, 3),
    })

results = pd.DataFrame(records)
display(results)


실행 결과는 Google Sheets 개발 기록에 질문 ID, 설정값, 검색 문서, 최종 답변, 응답 시간을 기록합니다. 숫자 계산과 표 기반 질문은 reference_answer와 직접 비교해 사람이 최종 판정합니다.
